# GEO Manipulation — Combined Edits (Notebook B)

Applies 11 combinatorial GEO methods to the target documents of 500 retail queries. Each GEO method combines multiple single edits (Fluency, Citations, Quotes, Statistics) and rewrites the target document using GPT-4o-mini. Since the manipulation is time-consuming due to the large number of LLM calls, the workload is split across four parallel notebooks (A–D), each processing its own query range.

**Input:**
- `data/retail/dataset/selected_docs.json` — target document index per query (from Puerto et al.)

**Output:**
- `data/retail/dataset/2_manipulation_selected_docs_{A/B/C/D}_v1.json` — subset file per notebook
- `data/retail/dataset/2_manipulation_selected_docs_v1.json` — merged result after all four notebooks complete

## Parallel Setup
| Notebook | Range |
|---|---|
| A | 0 – 124 |
| B | 125 – 249 |
| C | 250 – 374 |
| D | 375 – 499 |

## Structure
1. **Setup** — imports and paths
2. **Parameters** — notebook ID and query range
3. **Create Subset File** — creates the notebook-specific subset JSON
4. **Manipulation** — applies all 11 GEO methods to each target document
5. **Merge Results** — combines A–D back into the main file (run after all four complete)
6. **Summary** — completion status and manipulation checks

## Setup

In [ ]:
import json
import os
import sys
import time
from datetime import datetime

# Determine project root relative to this notebook's location
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

# Add src/ to path so we can import local modules (llms, methods)
sys.path.insert(0, os.path.join(project_root, "src"))

# Import OpenAI helper for LLM-based manipulation methods
from llms import OpenAIHelper

# Import the 11 combined GEO methods (2-way, 3-way, 4-way combinations of F, C, Q, S)
from methods import COMBINED_GEO_METHODS

# Load API key from config.json — never hardcode credentials
config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

# Data directory and input file
# selected_docs.json contains the target document per query — read-only input
data_dir = os.path.join(project_root, "data", "retail")
source_path = os.path.join(data_dir, "dataset", "selected_docs.json")

print("Setup complete.")
print(f"Project root: {project_root}")
print(f"Methods: {list(COMBINED_GEO_METHODS.keys())}")

## Parameters

**Change these values for each notebook instance.**

In [ ]:
# LLM used for all manipulation methods that require text generation
LLM_NAME = "gpt-4o-mini-2024-07-18"

# Each of the four parallel notebooks processes a separate range of queries
# A: 0–124 | B: 125–249 | C: 250–374 | D: 375–499
NOTEBOOK_ID = "B"   # A, B, C, or D
QUERY_START = 125   # inclusive
QUERY_END   = 149   # inclusive

# Output file for this notebook — each notebook writes to its own file
subset_path = os.path.join(data_dir, "dataset", f"2_manipulation_selected_docs_{NOTEBOOK_ID}_v1.json")

# 11 combinatorial edits of F, C, Q, S
METHOD_NAMES = list(COMBINED_GEO_METHODS.keys())

print(f"Notebook ID:     {NOTEBOOK_ID}")
print(f"Query range:     {QUERY_START} – {QUERY_END}")
print(f"Output file:     2_manipulation_selected_docs_{NOTEBOOK_ID}_v1.json")
print(f"LLM:             {LLM_NAME}")
print(f"Methods:         {METHOD_NAMES}")

## Create Subset File

Creates `2_manipulation_selected_docs_{ID}_v1.json` with only the queries in the defined range.
Skipped if the file already exists.

In [ ]:
# Skip creation if the subset file already exists
# This prevents overwriting work if the notebook is re-run
if os.path.exists(subset_path):
    print(f"2_manipulation_selected_docs_{NOTEBOOK_ID}_v1.json already exists — skipping creation.")
else:
    # Load selected_docs.json (Puerto et al.) containing all 500 queries
    with open(source_path, "r", encoding="utf-8") as f:
        all_docs = json.load(f)

    # Extract only the queries assigned to this notebook's range
    subset = {
        idx: all_docs[idx]
        for idx in all_docs
        if QUERY_START <= int(idx) <= QUERY_END
    }

    # Save the subset to a dedicated file for this notebook
    with open(subset_path, "w", encoding="utf-8") as f:
        json.dump(subset, f, indent=4, ensure_ascii=False)

    print(f"Created 2_manipulation_selected_docs_{NOTEBOOK_ID}_v1.json with {len(subset)} queries ({QUERY_START}–{QUERY_END})")

## Manipulation

Reads and writes only `2_manipulation_selected_docs_{ID}_v1.json` — no conflicts with other notebooks.
Safe to interrupt and resume.

In [ ]:
# Initialize the LLM client with the specified model
llm = OpenAIHelper(LLM_NAME)

# Load the subset of selected docs assigned to this notebook
with open(subset_path, "r", encoding="utf-8") as f:
    selected_docs = json.load(f)

# Sort query indices numerically for consistent processing order
query_indices = sorted(selected_docs.keys(), key=lambda x: int(x))
total = len(query_indices)

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print(f"Notebook {NOTEBOOK_ID}: queries {QUERY_START}–{QUERY_END} ({total} docs)")
print()

# Apply each GEO method to every target document in this notebook's range
for method_name in METHOD_NAMES:
    # Each method result is stored under the key "MethodName(doc)" in selected_docs
    key = f"{method_name}(doc)"

    # Count how many docs already have a result for this method
    # Allows safe re-runs without reprocessing completed entries
    already_done = sum(
        1 for idx in query_indices
        for entry in selected_docs[idx].values()
        if entry.get(key) is not None
    )

    # Skip this method entirely if all docs are already processed
    if already_done == total:
        print(f"[{method_name}] All {total} docs already done — skipping")
        continue
    
    # Instantiate the GEO method with the LLM client
    method = COMBINED_GEO_METHODS[method_name](llm)
    print(f"[{method_name}] Starting ({already_done}/{total} already done)")

    for i, query_idx in enumerate(query_indices):
        for doc_idx, entry in selected_docs[query_idx].items():

            # Skip if this doc already has a result for this method
            if entry.get(key) is not None:
                continue

            doc = entry["doc"]

            try:
                # Apply the GEO method to the document
                result = method.improve_text(doc)
                selected_docs[query_idx][doc_idx][key] = result
                
                # Save after every successful edit — ensures no work is lost on crash
                with open(subset_path, "w", encoding="utf-8") as f:
                    json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                print(f"  [{i+1}/{total}] query {query_idx}: done")

            except Exception as e:
                # First failure — wait 10 seconds and retry once
                print(f"  [{i+1}/{total}] query {query_idx}: ERROR — {e}")
                time.sleep(10)
                try:
                    result = method.improve_text(doc)
                    selected_docs[query_idx][doc_idx][key] = result
                    with open(subset_path, "w", encoding="utf-8") as f:
                        json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                    print(f"  [{i+1}/{total}] query {query_idx}: done (retry OK)")
                except Exception as e2:
                    # Second failure — store None to mark as failed, continue with next doc
                    print(f"  [{i+1}/{total}] query {query_idx}: FAILED — {e2}")
                    selected_docs[query_idx][doc_idx][key] = None

    # Count completed docs after finishing this method
    done = sum(
        1 for idx in query_indices
        for entry in selected_docs[idx].values()
        if entry.get(key) is not None
    )
    print(f"[{method_name}] Complete: {done}/{total}")
    print()

# Summary
end_time = datetime.now()
elapsed = end_time - start_time
print(f"{'='*60}")
print(f"NOTEBOOK {NOTEBOOK_ID} COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")

## Merge Results

Run this cell **only after all 4 notebooks are complete**.
Merges A, B, C, D back into `2_manipulation_selected_docs_v1.json`.

In [ ]:
NOTEBOOK_IDS = ["A", "B", "C", "D"]

# Merge all four notebook subsets into one combined selected_docs file
merged = {}
for nb_id in NOTEBOOK_IDS:
    path = os.path.join(data_dir, "dataset", f"2_manipulation_selected_docs_{nb_id}_v1.json")
    if not os.path.exists(path):
        print(f"WARNING: 2_manipulation_selected_docs_{nb_id}_v1.json not found — skipping.")
        continue
    with open(path, "r", encoding="utf-8") as f:
        subset = json.load(f)
    merged.update(subset)
    print(f"Loaded 2_manipulation_selected_docs_{nb_id}_v1.json ({len(subset)} queries)")

# Sort merged result by query index to ensure consistent ordering
merged = dict(sorted(merged.items(), key=lambda x: int(x[0])))

# Save the combined file as 2_manipulation_selected_docs_v1.json
output_path = os.path.join(data_dir, "dataset", "2_manipulation_selected_docs_v1.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=4, ensure_ascii=False)

print(f"\nMerged {len(merged)} queries into 2_manipulation_selected_docs_v1.json")

## Summary — Completion Status

In [ ]:
# Load the merged selected_docs file containing all 500 queries and all GEO method results
with open(os.path.join(data_dir, "dataset", "2_manipulation_selected_docs_v1.json"), "r") as f:
    all_docs = json.load(f)

total = len(all_docs)
print(f"RETAIL — full dataset ({total} docs)")
print()

# Print a progress bar for each combined GEO method
# Shows how many of the 500 target documents have been successfully edited
print("Combined methods:")
for method_name in METHOD_NAMES:
    key = f"{method_name}(doc)"
    
    # Count entries where the method result is not None
    done = sum(1 for q in all_docs.values() for e in q.values() if e.get(key) is not None)
    
    # Build ASCII progress bar — filled blocks represent completed docs
    bar = "\u2588" * (done * 20 // total) + "\u2591" * (20 - done * 20 // total)
    print(f"  {key:25} [{bar}] {done}/{total}")